# EZStats - Train Ball Detector v2  (#4)

**This one actually has new inputs:** your local ball set (`detector_ball_only`, **3956 images**) is v4 of the same Roboflow project the old model used at **~600 images**. 6x more data + better training = a real chance at higher recall (= ball found in more frames = fewer gaps/jumps).

| | Old model | This v2 |
|---|---|---|
| Data | ~600 imgs (v2) | **3956 imgs (v4) - uploaded by you** |
| Model | yolov8x | yolov8x (ball is tiny - keep big model) |
| Epochs | 50 | **80 + early stop** (more data needs fewer epochs) |
| Augmentation | tutorial default | **scale + HSV + mosaic + flip** |
| Class | `ball` (nc=1) | `ball` (nc=1) - same |

> Still: the 1-frame 'jump' is ultimately a *code* fix (temporal post-processing). Run #1 and #2 first; this is the lowest priority.

---
### BEFORE you start - upload your data (one time)
1. On your laptop, `ball.zip` is on your **Desktop** (Claude made it; ~208 MB).
2. Upload `ball.zip` anywhere inside your `ezstats/` folder on Drive (cell D auto-finds it).
3. Upload THIS notebook to `ezstats/` and open in Colab.

### Run order: A -> B -> C -> D -> E -> F -> G.

## A - Turn on the GPU
**Runtime -> Change runtime type -> GPU -> Save.** yolov8x at imgsz 1280 is heavy: T4 works but slow; A100 strongly preferred. Run the cell.

In [ ]:
!nvidia-smi

## B - Connect Google Drive (data read from here; model saves here, resume-safe)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path
HOME = os.getcwd()
RUNS_DIR = Path('/content/drive/MyDrive/ezstats/runs')
RUN_NAME = 'ball_detector_v2'
RUNS_DIR.mkdir(parents=True, exist_ok=True)
print('Saving to:', RUNS_DIR / RUN_NAME)

## C - Install

In [ ]:
!pip install -q ultralytics

## D - Unzip YOUR uploaded ball data + build a clean data.yaml
Auto-finds `ball.zip` and normalizes Windows backslash paths. Expect `train images: 3956` at the end - if it's 0, the zip is wrong.

In [ ]:
import zipfile, glob, shutil
from pathlib import Path

_cands = sorted(glob.glob('/content/drive/MyDrive/**/ball.zip', recursive=True))
assert _cands, 'ball.zip not found in MyDrive -> upload it into your ezstats folder first'
DATA_ZIP = Path(_cands[0])
print('Using data zip:', DATA_ZIP)

DATA_DIR = Path('/content/datasets/ball')
if DATA_DIR.exists():
    shutil.rmtree(DATA_DIR)
DATA_DIR.mkdir(parents=True, exist_ok=True)

# Normalize Windows backslash entry names -> real folders
with zipfile.ZipFile(DATA_ZIP) as z:
    for info in z.infolist():
        name = info.filename.replace('\\', '/')
        if name.endswith('/'):
            continue
        tgt = DATA_DIR / name
        tgt.parent.mkdir(parents=True, exist_ok=True)
        with z.open(info) as s, open(tgt, 'wb') as d:
            shutil.copyfileobj(s, d)

DATA_YAML = DATA_DIR / 'data.yaml'
DATA_YAML.write_text(
    f'train: {DATA_DIR}/train/images\n'
    f'val: {DATA_DIR}/valid/images\n'
    f'test: {DATA_DIR}/test/images\n'
    'nc: 1\n'
    "names: ['ball']\n"
)
print(DATA_YAML.read_text())
n_train = len(list((DATA_DIR / 'train' / 'images').glob('*')))
print('train images:', n_train)
assert n_train > 0, 'Extraction produced 0 images - check the zip structure'

## E - Train (resume-safe)
~5-8 h on T4, ~1.5 h on A100 for 80 epochs on 3956 imgs. Lower to `epochs=50` for a quick test. Re-run after any disconnect to resume.

In [ ]:
%cd {HOME}

ckpt = RUNS_DIR / RUN_NAME / 'weights' / 'last.pt'
if ckpt.exists():
    print('Found checkpoint -> RESUMING from', ckpt)
    !yolo task=detect mode=train resume=True model='{ckpt}'
else:
    print('Fresh training run.')
    !yolo task=detect mode=train \
      model=yolov8x.pt \
      data='{DATA_YAML}' \
      epochs=80 \
      imgsz=1280 \
      batch=8 \
      patience=25 \
      cos_lr=True \
      close_mosaic=10 \
      hsv_h=0.015 hsv_s=0.7 hsv_v=0.4 \
      translate=0.1 scale=0.5 fliplr=0.5 \
      mosaic=1.0 \
      plots=True \
      project='{RUNS_DIR}' name='{RUN_NAME}'

## F - Check results
The old model scored mAP50=0.932, **recall=0.832**. You want to BEAT recall=0.83 - higher recall = ball found in more frames. If it doesn't beat the old recall, don't swap it in.

In [ ]:
%cd {HOME}
!yolo task=detect mode=val \
  model='{RUNS_DIR}/{RUN_NAME}/weights/best.pt' \
  data='{DATA_YAML}' imgsz=1280

In [ ]:
from IPython.display import Image
Image(filename=f'{RUNS_DIR}/{RUN_NAME}/results.png', width=900)

## G - Done - model on Drive
`MyDrive/ezstats/runs/ball_detector_v2/weights/best.pt`

On laptop: download -> `artifacts/ball/football-ball-detection-v2.pt` (NEW name, keep old). Tell Claude to validate before any swap.